# SAE "Analyse automatique de données" -- Notebook 4 : régression linéaire

Dans ce notebook, vous devez :
1. Effectuer des expérimentations pour optimiser les paramètres du modèle de régression linéaire ;
2. Rédiger une synthèse pour justifier des pré-traitements effectués et des valeurs des paramètres obtenus pour vos modèles de régression. Vous concluerez sur le meilleur choix des pré-traitements et des paramètres de votre ou vos modèles.

	
Vous pouvez également tout d'abord considérer un sous-ensemble de votre ensemble d'apprentissage, pour mettre au point chacune de vos expérimentations.
			


## Modules python utilisés

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score 
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
import math
import spacy
import numpy as np


## Utils

In [ ]:
def displayMetriquesAndGraph(test, prediction) :
    """
    Affiche les différentes métriques et graphiques
    """
    mse = mean_squared_error(test,prediction)

    print(f"Mse: {mse}")
    print(f"RMSE: {math.sqrt(mse)}")
    print(f"R2 Score: {r2_score(test,prediction)}")
    plt.scatter(test, prediction, color='blue', alpha=0.1)

    plt.plot([1, 5], [1, 5], color='red', linestyle='--', linewidth=2, label='Idéal')

    plt.title('Prédictions vs Réalité')
    plt.xlabel('Vraies Notes')
    plt.ylabel('Notes Prédites')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    
def findBestRmse(result: dict):
    """
    Renvoi la valeur maximum de la RMSE
    """
    key_value = ""
    max = 10
    for key,val in result.items():
        if (val['rmse'] < max):
            key_value = key
            max = val['rmse']
    return key_value; 


def findBestR2(result: dict):
    """
    Renvoi la valeur minimum de R2
    """
    key_value = ""
    max = 0
    for key,val in result.items():
        if (val['r2'] > max):
            key_value = key
            max = val['r2']
    return key_value; 


def isolateCategory(category : str, x_train_local,x_test_local,y_train_local,y_test_local):
    """
    Isole une catégorie par son nom dans les dataframes d'entrainement et de test
    """
    cat_x_train = x_train_local[x_train_local['category'] == category]
    cat_x_test = x_test_local[x_test_local['category'] == category]
    cat_y_train = y_train_local.loc[cat_x_train.index]
    cat_y_test = y_test_local.loc[cat_x_test.index]
    
    return cat_x_train, cat_x_test, cat_y_train, cat_y_test

In [ ]:
## Utils - GridSearchCV

def calculateHyperParameters(model, x_train_local,y_train_local,x_test_local,y_test_local):
    """
    Calcul les meilleurs paramètres, lance les prédiction et affiche les résultats
    """
    model.fit(x_train_local, y_train_local)

    print(f"Meilleurs paramètres : {model.best_params_}")
    print(f"Meilleure RMSE : {-model.best_score_:.3f}")

    prediction_linear_regression = model.best_estimator_.predict(x_test_local)

    displayMetriquesAndGraph(y_test_local,prediction_linear_regression)

## 1. Régression linéaire

In [ ]:
#Df d'apprentissage
x_train = pd.read_csv('data/train_test/x_train.csv',  index_col=0)
y_train = pd.read_csv('data/train_test/y_train.csv',  index_col=0)

#Df de test
x_test = pd.read_csv('data/train_test/x_test.csv',  index_col=0)
y_test = pd.read_csv('data/train_test/y_test.csv',  index_col=0)


In [ ]:
x_train = x_train.dropna(subset=['text_']).copy()
y_train = y_train.loc[x_train.index].copy()

x_test = x_test.dropna(subset=['text_']).copy()
y_test = y_test.loc[x_test.index].copy()

x_train['text_'] = x_train['text_'].astype(str)
x_test['text_'] = x_test['text_'].astype(str)

nlp = spacy.load("en_core_web_sm") # Charger le package en_core_web_sm

def cleanData(texts):
    processed_texts = []
    for doc in nlp.pipe(texts):
        try:
            # On extrait la version lemmatisé
            tokens = [
                token.lemma_.lower() for token in doc
                if not token.is_stop and not token.is_punct  # Enlever les mots vides (that, is, the...) et les signes de ponctuations
                and token.pos_ in ["ADJ","ADV","VERB","NOUN"] # Garder les mots identifés comme Adverbes, adverbes, verbe, nom
            ]
            processed_texts.append(" ".join(tokens))
        except Exception as e :
            print(f"Erreur sur la phrase : {e}")
    return processed_texts
    
x_train['text_'] = cleanData(x_train['text_'])
x_test['text_'] = cleanData(x_test['text_'])

# Supprimer les lignes vides
x_train['text_'] = x_train['text_'].replace("", np.nan) 
x_test['text_'] = x_test['text_'].replace("", np.nan)

x_train = x_train.dropna(subset=['text_']).copy()
x_test = x_test.dropna(subset=['text_']).copy()

# Mettre à jour les df de résultats
y_train = y_train.loc[x_train.index]
y_test = y_test.loc[x_test.index]

### 1.1 Régression avec le texte - Linear Regression

In [ ]:
# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=2000, # Garder les X termes plus importants
            min_df=100, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_'),
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', LinearRegression())
])

param_grid = {
    'preprocessor__review__max_features': [2000, 5000, 10000],
    'preprocessor__review__min_df': [5, 10,50,100,500],
}

reg_search_linear_regression = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

calculateHyperParameters(reg_search_linear_regression,
                         x_train_local= x_train,
                         x_test_local=x_test,
                         y_train_local=y_train,
                         y_test_local=y_test)


### Régression avec toutes les catégories + texte

In [ ]:
# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=2000, # Garder les X termes plus importants
            min_df=100, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_'),
        ('category',OneHotEncoder(),['label'])
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', LinearRegression())
])

param_grid = {
    'preprocessor__review__max_features': [2000, 5000, 10000],
    'preprocessor__review__min_df': [5, 10,50,100,500],
}

reg_search_linear_regression = GridSearchCV( model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

calculateHyperParameters(reg_search_linear_regression,
                         x_train_local= x_train,
                         x_test_local=x_test,
                         y_train_local=y_train,
                         y_test_local=y_test)


### Régression par catégorie - LinearRegression

In [ ]:
# Isole toutes les catégories
categories = x_train['category'].unique() 


# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=100, # Garder les X termes plus importants
            min_df=50, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_')
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', LinearRegression())
])

param_grid = {
    'preprocessor__review__max_features': [50,100,500,1000,2000],
    'preprocessor__review__min_df': [2, 10, 0.01, 0.05],
}

reg_search_linear_regression = GridSearchCV(
    model, 
    param_grid, 
    cv=5,                   # Validation croisée en 5 plis
    scoring='neg_mean_squared_error', # Score pour la régression
    n_jobs=-1,              # Utilise tous les processeurs
    verbose=1,
    error_score=np.nan
)

# Dictionnaire des résultats contenant la liste des résultats de chaque catégorie
results = {}

for category in categories:
    print(f"Lancement pour la catégorie {category}")
    cat_x_train, cat_x_test, cat_y_train, cat_y_test = isolateCategory(category,x_train,x_test,y_train,y_test)

    reg_search_linear_regression.fit(cat_x_train, cat_y_train)

    print(f"Meilleurs paramètres : {reg_search_linear_regression.best_params_}")
    print(f"Meilleure RMSE : {-reg_search_linear_regression.best_score_:.3f}")

    prediction_linear_regression = reg_search_linear_regression.best_estimator_.predict(cat_x_test) 

    # Ajout des résultats
    mse = mean_squared_error(cat_y_test,prediction_linear_regression)
    r2 = r2_score(cat_y_test,prediction_linear_regression)
    rmse = math.sqrt(mse)
    
    print(f"Mse: {mse}")
    print(f"RMSE: {rmse}")
    print(f"R2 Score: {r2}")
    plt.scatter(cat_y_test, prediction_linear_regression, color='blue', alpha=0.1)

    plt.plot([1, 5], [1, 5], color='red', linestyle='--', linewidth=2, label='Idéal')

    plt.title(f'Prédictions vs Réalité : {category}')
    plt.xlabel('Vraies Notes')
    plt.ylabel('Notes Prédites')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
    results[category] = {"mse":mse,"rmse":rmse,"r2":r2}
    print("-------------------------")
    
# Analyse des résultats



print("Résultats finaux")
bestRmse = findBestRmse(results)
bestR2 = findBestR2(results)
print(f"Meilleure RMSE: {results[bestRmse]['rmse']} pour la catégorie {bestRmse}")

print(f"Meilleur score r2: {results[bestR2]['r2']} pour la catégorie {bestR2}")


### 1.3 Régression avec le texte - Ridge

In [ ]:
# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=2000, # Garder les X termes plus importants
            min_df=10, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_'),
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', Ridge(alpha=10))
])

param_grid = {
    'preprocessor__review__max_features': [2000, 5000, 10000],
    'preprocessor__review__min_df': [5, 10,50,100,500],
    'classifier__alpha':[0.001, 0.01, 0.1, 0, 1, 10]
}

reg_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1,verbose=1)

calculateHyperParameters(reg_search,
                         x_train_local= x_train,
                         x_test_local=x_test,
                         y_train_local=y_train,
                         y_test_local=y_test)

### 1.3 Texte + catégorie - Ridge

In [ ]:
# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=2000, # Garder les X termes plus importants
            min_df=10, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_'),
        ('category',OneHotEncoder(),['label'])
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', Ridge(alpha=10))
])

param_grid = {
    'preprocessor__review__max_features': [2000, 5000, 10000],
    'preprocessor__review__min_df': [5, 10,50,100,500],
    'classifier__alpha':[0.001, 0.01, 0.1, 0, 1, 10]
}

reg_search = GridSearchCV(model, param_grid, cv=5,scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

calculateHyperParameters(reg_search,
                         x_train_local= x_train,
                         x_test_local=x_test,
                         y_train_local=y_train,
                         y_test_local=y_test)

### Ridge régression par catégories


In [ ]:
# Isole toutes les catégories
categories = x_train['category'].unique() 


# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=100, # Garder les X termes plus importants
            min_df=50, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_')
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', Ridge())
])

param_grid = {
    'preprocessor__review__max_features': [50,100,500,1000,2000],
    'preprocessor__review__min_df': [2, 10, 0.01, 0.05],
    'classifier__alpha':[0.001, 0.01, 0.1, 0, 1, 10]
}

reg_search_linear_regression = GridSearchCV(
    model, 
    param_grid, 
    cv=5,                   # Validation croisée en 5 plis
    scoring='neg_mean_squared_error', # Score pour la régression
    n_jobs=-1,              # Utilise tous les processeurs
    verbose=1,
    error_score=np.nan
)

# Dictionnaire des résultats contenant la liste des résultats de chaque catégorie
results = {}

for category in categories:
    print(f"Lancement pour la catégorie {category}")
    cat_x_train, cat_x_test, cat_y_train, cat_y_test = isolateCategory(category,x_train,x_test,y_train,y_test)

    reg_search_linear_regression.fit(cat_x_train, cat_y_train)

    print(f"Meilleurs paramètres : {reg_search_linear_regression.best_params_}")
    print(f"Meilleure RMSE : {-reg_search_linear_regression.best_score_:.3f}")

    prediction_linear_regression = reg_search_linear_regression.best_estimator_.predict(cat_x_test) 

    # Ajout des résultats
    mse = mean_squared_error(cat_y_test,prediction_linear_regression)
    r2 = r2_score(cat_y_test,prediction_linear_regression)
    rmse = math.sqrt(mse)
    
    print(f"Mse: {mse}")
    print(f"RMSE: {rmse}")
    print(f"R2 Score: {r2}")
    plt.scatter(cat_y_test, prediction_linear_regression, color='blue', alpha=0.1)

    plt.plot([1, 5], [1, 5], color='red', linestyle='--', linewidth=2, label='Idéal')

    plt.title(f'Prédictions vs Réalité : {category}')
    plt.xlabel('Vraies Notes')
    plt.ylabel('Notes Prédites')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.show()
    results[category] = {"mse":mse,"rmse":rmse,"r2":r2}
    print("-------------------------")
    
# Analyse des résultats
print("Résultats finaux")
bestRmse = findBestRmse(results)
bestR2 = findBestR2(results)
print(f"Meilleure RMSE: {results[bestRmse]['rmse']} pour la catégorie {bestRmse}")

print(f"Meilleur score r2: {results[bestR2]['r2']} pour la catégorie {bestR2}")


### 1.4 Lasso

In [22]:
# On défini les différents transformeurs 
processeur = ColumnTransformer(
    remainder='drop',
    transformers=[
        ('review', TfidfVectorizer(
            max_features=2000, # Garder les X termes plus importants
            min_df=10, # Ignorer les mots avec une rareté inférieur à 20
            ngram_range=(1,2), # Capte les expressions de 2 mots monogramme et bigramme
            stop_words='english',
            use_idf=True), 
         'text_'),
    ]
)

print(f"Processeur : {processeur.remainder}")

model = Pipeline([
    ('preprocessor', processeur),
    ('classifier', Lasso(alpha=0.0001))
])

param_grid = {
    'preprocessor__review__max_features': [2000, 5000, 10000],
    'preprocessor__review__min_df': [5, 10,50,100,500],
    'classifier__alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1, 10]
}

reg_search = GridSearchCV(model, param_grid, cv=5,scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

calculateHyperParameters(reg_search,
                         x_train_local= x_train,
                         x_test_local=x_test,
                         y_train_local=y_train,
                         y_test_local=y_test)


Processeur : drop


NameError: name 'calculateHyperParameters' is not defined

## 2. Synthèse des expérimentations et des résultats obtenus

### Protocole d'expérimentations

Pour chacune des expérimentations la fonction `GridSearchCV` est utilisé afin de trouver les meilleurs paramètres, elle à pour instruction de se baser sur la `neg_mean_squared_error`, on utilise `neg` car par défaut la fonction cherche à obtenir la valeur la plus haute alors qu'ici on cherche la valeur la plus basse, on utilise la validation croisée avec `5` itérations pour obtenir une moyenne avec du sens.

Une fois les meilleurs paramètres trouvés on réalise l'entrainement et les prédictions sur ces derniers, enfin on affiche les métriques et les graphiques correspondant.

### Tableaux des paramètres

| Nom                        | max_features | min_df | ngram_ranges | stop_words | use_idf |
| -------------------------- | ------------ | ------ | ------------ | ---------- | ------- |
| Reg text toutes catégories | 2000         | 100    | 1,2          | english    | True    |
| Reg text + catégories      | 2000         | 100    |              | english    | True    |
| Reg text par catégories    | 50           | 0.01   | 1,2          | english    | True    |

### Tableaux des résultats

| Nom                        | MSE  | RMSE | R2   | Notes                      |
| -------------------------- | ---- | ---- | ---- | -------------------------- |
| Reg text toutes catégories | 1.5  | 1.22 | 0.16 |                            |
| Reg text + catégories      | 1.49 | 1.22 | 0.16 |                            |
| Reg text par catégories    | /    | 1.10 | 0.07 | catégories: Kindle_Store_5 |

_Notes_: Les résultats de l'expérimentation `Reg text par catégories` sont ceux de la catégorie avec la meilleure RMSE, ici Kindle_Store_5.

**Conclusion:**
Malgré une légère amélioration par la segmentation des données, la régression linéaire se montre insufisante pour modéliser la complexité des avis textuels,
La RMSE ne descend pas en dessous pas de 1.10.

# Partie bonus

Dans cette partie nous avons utilisé le classifieur `Ridge` et `Lasso`

### Tableaux des paramètres

| Nom                           | max_features | min_df | ngram_ranges | alpha  | stop_words | use_idf |
| ----------------------------- | ------------ | ------ | ------------ | ------ | ---------- | ------- |
| Reg texte - Ridge             | 2000         | 10     | 1,2          | 10     | english    | True    |
| Reg texte + categorie - Ridge | 2000         | 10     | 1,2          | 10     | english    | True    |
| Reg text par catégories       | 2000         | 2      | 1,2          | 1      | english    | True    |
| Reg texte - Lasso             | 2000         | 10     | 1,2          | 0.0001 | english    | True    |

### Tableaux des résultats

| Nom                              | MSE  | RMSE | R2   | Notes                     |
| -------------------------------- | ---- | ---- | ---- | ------------------------- |
| Reg texte - Ridge                | 1.46 | 1.21 | 0.18 |                           |
| Reg texte + catégorie - Ridge    | 1.44 | 1.2  | 0.2  |                           |
| Reg texte par catégories - Ridge | 1.28 | 1.13 | 0.01 | catégorie: Kindle_Store_5 |
| Reg texte - Lasso                | 1.49 | 1.22 | 0.17 |                           |

**Conclusion**:
L'utilisation du modèle `Ridge` et `Lasso` nous permet de gagner en précision lors de la régression
